# LAB 5a:  Vertex AI での Keras モデルのトレーニング

**学習目標**

1. 環境をセットアップする
1. ハイパーパラメータの引数解析（argparsing）コードを保持するトレーナー モジュールの task.py を作成する
1. Keras モデルのコードを保持するトレーナー モジュールの model.py を作成する
1. トレーナー モジュール パッケージをローカルで実行する
1. Vertex AI にトレーニング ジョブを送信する


## はじめに 
このノートブックでは、以前に開発した Keras モデルコードを取得し、それを Python モジュールとしてパッケージ化して、Vertex AI でトレーニングします。

まず、環境をセットアップし、`cloudml-hypertune` パッケージをインポートします。次に、トレーナー モジュールの `task.py` を作成して、ハイパーパラメータの引数解析ロジックを保持します。続いて、Keras モデルコードを保持する `model.py` を作成します。その後、小規模なデータセットでトレーナー モジュールをローカルで実行し、すべてが想定どおりに動作することを確認します。ローカルで動作することを確認したら、コードをソース ディストリビューションとしてパッケージ化し、Google Cloud Storage にアップロードして、Vertex AI カスタム トレーニング ジョブを送信します。最後に、ハイパーパラメータ チューニング ジョブを実行して最適なハイパーパラメータを見つけ、それらを使用してトレーニングを繰り返します。

各学習目標は、この受講者用ラボ ノートブック内の __#TODO__ に対応しています。まずこのノートブックを完了してみてから、[解答ノートブック](../solutions/5a_train_keras_vertex_babyweight.ipynb) を確認してください。

## 環境変数の設定と必要なライブラリの読み込み

必要なライブラリをインポートしましょう。特に、`cloudml-hypertune` パッケージをインポートします。これは、ハイパーパラメータ チューニングの指標を Vertex AI にレポートするために使用するパッケージです。このパッケージをインポートすることで、トレーナー パッケージをローカルでテストできるようになります。

In [ ]:
import os

import hypertune

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

### 環境変数の設定

ラボ全体で使用できるように環境変数を設定します。バケット名にはプロジェクト名を使用します。

In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

os.environ["PROJECT"] = PROJECT
os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

バケットが存在しない場合は作成し、以下でバケットがリージョナルであり、そのリージョンが指定されたリージョンと一致することを確認します:

In [ ]:
 %%bash
if ! gcloud storage ls | grep -q gs://${BUCKET}/; then
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
fi
gcloud storage ls -L -b gs://$BUCKET | grep "gs://\|Location"
echo $REGION

In [ ]:
%%bash
gcloud config set project ${PROJECT}
gcloud config set ai/region ${REGION}

## データが存在することの確認

トレーニングと評価に使用する CSV ファイルが以前に作成されていることを確認します。作成されていない場合は、ラボ [1b_prepare_data_babyweight](../solutions/1b_prepare_data_babyweight.ipynb) に戻って作成してください。

In [ ]:
%%bash
gcloud storage ls gs://${BUCKET}/babyweight/data/*000000000000.csv

データのサブセットで [Keras Wide & Deep のコード](../solutions/4c_keras_wide_and_deep_babyweight.ipynb) が動作するようになったので、TensorFlow コードを Python モジュールとしてパッケージ化し、Vertex AI でトレーニングできます。

## Vertex AI でのトレーニング

Vertex AI でのトレーニングには以下が必要です:
* コードを Python ソース ディストリビューションにする
* gcloud を使用してトレーニング コードを [Vertex AI](https://console.cloud.google.com/vertex-ai) に送信する

この [リンク](https://console.developers.google.com/apis/library/aiplatform.googleapis.com) にアクセスして、Vertex AI API が有効になっていることを確認してください。

### コードを Python パッケージに移動する

Python パッケージとは、ディレクトリをパッケージとして識別するための `__init__.py` ファイルとともに、1 つ以上の `.py` ファイルを集めたものです。`__init__.py` には初期化コードが含まれることもありますが、ここでの目的には空のファイルで十分です。

bash コマンド `touch` は指定された場所に空のファイルを作成します。ディレクトリ `babyweight` はすでに存在しているはずです。

In [ ]:
%%bash
mkdir -p babyweight/trainer
touch babyweight/trainer/__init__.py

次に、`%%writefile` マジックを使用して、下のセルの内容を `babyweight/trainer` フォルダ内の `task.py` というファイルに書き込みます。

### ラボ課題 1: ハイパーパラメータの引数解析コードを保持するトレーナー モジュールの task.py を作成する

下のセルは、トレーニング ジョブを設定するファイル `babyweight/trainer/task.py` を書き出します。ここでは、`parser` モジュールを使用して、トレーニング中にフラグとして渡すモデルのパラメータを決定します。以下のコードで `batch_size` がモデルにどのように渡されるかを確認してください。これを例として使用して、次の変数の引数を解析してください。
- `nnsize`: DNN 特徴量列に使用する隠れレイヤのサイズ
- `nembeds`: n 個のキーとなる実数値パラメータの交差（cross）の埋め込みサイズ
- `train_examples`: トレーニング ジョブを実行するサンプル数（千単位）
- `eval_steps`: モデルを評価する正のステップ数

上記の解析対象引数には必ずデフォルト値を含め、必要に応じて `type` を指定してください。

In [ ]:
%%writefile babyweight/trainer/task.py
import argparse
import json
import os

from trainer import model

import tensorflow as tf

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--train_data_path",
        help="GCS location of training data",
        required=True
    )
    parser.add_argument(
        "--eval_data_path",
        help="GCS location of evaluation data",
        required=True
    )
    parser.add_argument(
        "--output_dir",
        help="GCS location to write checkpoints and export models",
        default = os.getenv("AIP_MODEL_DIR")
    )
    parser.add_argument(
        "--batch_size",
        help="Number of examples to compute gradient over.",
        type=int,
        default=512
    )
    # TODO: Add nnsize argument

    # TODO: Add nembeds argument

    # TODO: Add num_epochs argument

    # TODO: Add train_examples argument

    # TODO: Add eval_steps argument


    # すべての引数をパース
    args = parser.parse_args()
    arguments = args.__dict__

    # 一部の引数を変更
    arguments["train_examples"] *= 1000

    # トレーニング ジョブの実行
    model.train_and_evaluate(arguments)

同様に、前のノートブックで開発したモデルをファイル `model.py` に書き込むことができます。 

### ラボ課題 2: Keras モデルコードを保持するトレーナー モジュールの model.py を作成する

下のコードセルの TODO を完了して、`model.py` を作成してください。Wide & Deep モデル用に作成したコードを使用します。[4c_keras_wide_and_deep_babyweight](../solutions/4c_keras_wide_and_deep_babyweight.ipynb) ノートブックを振り返り、そのノートブックから必要なコードをコピーして、下のセルの該当箇所に貼り付けてください。

In [ ]:
%%writefile babyweight/trainer/model.py
import datetime
import os
import shutil
import numpy as np
import tensorflow as tf
import hypertune

# CSV、ラベル、キー列を決定
# TODO: Add CSV_COLUMNS and LABEL_COLUMN

# TODO: Add NUMERIC_COLUMNS and CATEGORICAL_COLUMNS

# 各 CSV 列のデフォルト値を設定。
# is_male と plurality を文字列として扱います。
# TODO: Add DEFAULTS



def features_and_labels(row_data):
    # TODO: Add your code here
    pass


def load_dataset(pattern, batch_size=1, mode=tf.estimator.ModeKeys.EVAL):
    # TODO: Add your code here
    pass


def create_input_layers():
    # TODO: Add your code here
    pass


def transform(inputs, nembeds):
    # TODO: Add your code here
    pass

def get_model_outputs(wide_inputs, deep_inputs, dnn_hidden_units):
    # TODO: Add your code here
    pass



def rmse(y_true, y_pred):
    """Calculates RMSE evaluation metric.

    Args:
        y_true: tensor, true labels.
        y_pred: tensor, predicted labels.
    Returns:
        Tensor with value of RMSE between true and predicted labels.
    """
    return tf.sqrt(tf.reduce_mean(tf.square(y_pred - y_true)))


def build_wide_deep_model(dnn_hidden_units=[64, 32], nembeds=3):
    # TODO: Add your code here
    pass



# HyperTune レポーティング オブジェクトをインスタンス化
hpt = hypertune.HyperTune()

# レポーティング コールバック
class HPTCallback(tf.keras.callbacks.Callback):

    def on_epoch_end(self, epoch, logs=None):
        global hpt
        hpt.report_hyperparameter_tuning_metric(
            hyperparameter_metric_tag='val_rmse',
            metric_value=logs['val_rmse'],
            global_step=epoch)

        
def train_and_evaluate(args):
    model = build_wide_deep_model(args["nnsize"], args["nembeds"])
    print("Here is our Wide-and-Deep architecture so far:\n")
    print(model.summary())

    trainds = load_dataset(
        args["train_data_path"],
        args["batch_size"],
        tf.estimator.ModeKeys.TRAIN)

    evalds = load_dataset(
        args["eval_data_path"], 1000, tf.estimator.ModeKeys.EVAL)
    if args["eval_steps"]:
        evalds = evalds.take(count=args["eval_steps"])

    num_batches = args["batch_size"] * args["num_epochs"]
    steps_per_epoch = args["train_examples"] // num_batches

    checkpoint_path = os.path.join(args["output_dir"], "checkpoints/babyweight")
    cp_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path, verbose=1, save_weights_only=True)

    history = model.fit(
        trainds,
        validation_data=evalds,
        epochs=args["num_epochs"],
        steps_per_epoch=steps_per_epoch,
        verbose=2,  # 0=サイレント, 1=進行状況バー, 2=エポックごとに 1 行
        callbacks=[cp_callback, HPTCallback()])

    EXPORT_PATH = os.path.join(
        args["output_dir"], datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
    tf.saved_model.save(
        obj=model, export_dir=EXPORT_PATH)  # デフォルトのサービング関数を使用
    
    print("Exported trained model to {}".format(EXPORT_PATH))

## ローカルでのトレーニング

コードをパッケージに移動したら、スタンドアロンで動作することを確認します。パイプラインの開発中にデータセット全体でトレーニングを試みないように、`--train_examples` フラグを組み込んだことに注目してください。サブセットですべてが正常に動作することを確認したら、すべてのデータでトレーニングできるように変更できます。このサブセットであっても、出力が表示されない状態で約 *3 分間* かかります...

### ラボ課題 3: トレーナー モジュール パッケージをローカルで実行する

以下の TODO の不足しているコードを埋めて、小さなバッチサイズ、1 エポック、1 つのトレーニング サンプル、および 1 つの評価ステップを指定して、単一ファイルに対して非常に小規模なトレーニング ジョブを実行できるようにしてください。

In [ ]:
%%bash
OUTDIR=babyweight_trained
rm -rf ${OUTDIR}
export PYTHONPATH=${PYTHONPATH}:${PWD}/babyweight
python3 -m trainer.task \
    --train_data_path=gs://${BUCKET}/babyweight/data/train*.csv \
    --eval_data_path=gs://${BUCKET}/babyweight/data/eval*.csv \
    --output_dir=${OUTDIR} \
    --batch_size=# TODO: Add batch size
    --num_epochs=# TODO: Add the number of epochs to train for
    --train_examples=# TODO: Add the number of examples to train each epoch for
    --eval_steps=# TODO: Add the number of evaluation batches to run

## ラボ課題 4: Vertex AI でのトレーニング

ローカルですべてが動作することを確認できたので、クラウドでトレーニングしましょう！まず、コードをソース ディストリビューションとしてパッケージ化する必要があります。これには `setuptools` を使用できます。

In [ ]:
%%writefile babyweight/setup.py
from setuptools import find_packages
from setuptools import setup

setup(
    name='babyweight_trainer',
    version='0.1',
    packages=find_packages(),
    include_package_data=True,
    description='Babyweight model training application.'
)

In [ ]:
%%bash
cd babyweight
python ./setup.py sdist --formats=gztar
cd ..

パッケージを Cloud Storage バケットに保存します。

In [ ]:
%%bash
gcloud storage cp babyweight/dist/babyweight_trainer-0.1.tar.gz gs://${BUCKET}/babyweight/

Cloud に送信するには [`gcloud custom-jobs create`](https://cloud.google.com/sdk/gcloud/reference/ai/custom-jobs/create) を使用し、Vertex AI Training Service 用のいくつかの追加パラメータを指定するだけです:
- display-name: Cloud ジョブの一意の識別子。一意性を確保するために通常はシステム時間を追加します
- region: トレーニングを行うクラウド リージョン。サポートされている Vertex AI Training Service リージョンについては、[こちら](https://cloud.google.com/vertex-ai/docs/general/locations) を参照してください

コマンド呼び出しで `worker pool spec` とパススルー Python 引数を直接指定して実行された `gcloud ai custom-jobs create` を以前に見かけたことがあるかもしれませんが、ここでは YAML ファイルを使用します。これにより、ハイパーパラメータ チューニングへの移行が容易になります。

`args:` 引数を通じて、`task.py` ファイル用のパススルー引数を追加します。

#TODO を完了して、task.py のパーサーに必要な user_args が設定されていることを確認してください。

In [ ]:
%%bash

TIMESTAMP=$(date -u +%Y%m%d_%H%M%S)
OUTDIR=gs://${BUCKET}/babyweight/trained_model_$TIMESTAMP
JOB_NAME=babyweight_$TIMESTAMP

PYTHON_PACKAGE_URI=gs://${BUCKET}/babyweight/babyweight_trainer-0.1.tar.gz
PYTHON_PACKAGE_EXECUTOR_IMAGE_URI="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-8:latest"
PYTHON_MODULE=trainer.task

echo > ./config.yaml "workerPoolSpecs:
  machineSpec:
    machineType: n1-standard-4
  replicaCount: 1
  pythonPackageSpec:
    executorImageUri: $PYTHON_PACKAGE_EXECUTOR_IMAGE_URI
    packageUris: $PYTHON_PACKAGE_URI
    pythonModule: $PYTHON_MODULE
    args:
    - --train_data_path=# TODO: Add path to training data in GCS
    - --eval_data_path=# TODO: Add path to evaluation data in GCS
    - --output_dir=$OUTDIR
    - --num_epochs=# TODO: Add the number of epochs to train for
    - --train_examples=# TODO: Add the number of examples to train each epoch for
    - --eval_steps=# TODO: Add the number of evaluation batches to run
    - --batch_size=# TODO: Add batch size
    - --nembeds=# TODO: Add number of embedding dimensions

gcloud ai custom-jobs create \
  --region=${REGION} \
  --display-name=$JOB_NAME \
  --config=config.yaml

トレーニング ジョブは 10〜15 分以内に完了するはずです。次のラボを完了するにはトレーニング済みモデルが必要になります。

## ラボ課題 5: ハイパーパラメータ チューニング

ハイパーパラメータ チューニングを実行するには、YAML ファイルを作成し、その名前を `--config` で渡します。
このステップには<b>数時間</b>かかる場合があります。`--parallel-trial-count` を増やすか、`--max-trial-count` を減らすことで、より早く完了させることができます。`--parallel-trial-count` は探索を開始する初期シードの数であるため、あまり大きくしすぎないようにしてください。そうしないと、単なるランダム探索になってしまいます。

yaml ファイルと gcloud トレーニング ジョブの bash コマンド内の #TODO を完了して、ハイパーパラメータ チューニングを実行できるようにしてください。

In [ ]:
%%bash
TIMESTAMP=$(date -u +%Y%m%d_%H%M%S)
BASE_OUTPUT_DIR=gs://${BUCKET}/babyweight/hp_tuning_$TIMESTAMP
JOB_NAME=babyweight_hpt_$TIMESTAMP

PYTHON_PACKAGE_URI=gs://${BUCKET}/babyweight/babyweight_trainer-0.1.tar.gz
PYTHON_PACKAGE_EXECUTOR_IMAGE_URI="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-8:latest"
PYTHON_MODULE=trainer.task

echo > ./hyperparam.yaml "displayName: $JOB_NAME
studySpec:
  metrics:
  - metricId: # TODO: Add metric we want to optimize
    goal: # TODO: MAXIMIZE or MINIMIZE?
  parameters:
  - parameterId: batch_size
    # TODO: What datatype (which ValueSpec)?
      minValue: # TODO: Choose a min value
      maxValue: # TODO: Choose a max value
    scaleType: # TODO: UNIT_LINEAR_SCALE or UNIT_LOG_SCALE?
  - parameterId: nembeds
    # TODO: What datatype (which ValueSpec)?
      minValue: # TODO: Choose a min value
      maxValue: # TODO: Choose a max value
    scaleType: # TODO: UNIT_LINEAR_SCALE or UNIT_LOG_SCALE?
  algorithm: ALGORITHM_UNSPECIFIED # ベイズ最適化が適用されます
trialJobSpec:
  baseOutputDirectory:
    outputUriPrefix: $BASE_OUTPUT_DIR
  workerPoolSpecs:
  - machineSpec:
      machineType: n1-standard-8
    pythonPackageSpec:
      executorImageUri: $PYTHON_PACKAGE_EXECUTOR_IMAGE_URI
      packageUris:
      - $PYTHON_PACKAGE_URI
      pythonModule: $PYTHON_MODULE
      args:
      - --train_data_path=gs://${BUCKET}/babyweight/data/train*.csv
      - --eval_data_path=gs://${BUCKET}/babyweight/data/eval*.csv
      - --num_epochs=10
      - --train_examples=5000
      - --eval_steps=100
      - --batch_size=32
      - --nembeds=8
    replicaCount: 1"
        
gcloud beta ai hp-tuning-jobs create \
    --region=$REGION \
    --display-name=$JOB_NAME \
    --# TODO: Add config for hyperparam.yaml
    --max-trial-count=20 \
    --parallel-trial-count=5

## トレーニングの繰り返し

今回は、チューニングされた `batch_size` と `nembeds` のパラメータを使用してトレーニングを繰り返します。最良の結果は以下と異なる場合があることに注意してください。ご自身の結果を必ず記入してください！

In [ ]:
%%bash
TIMESTAMP=$(date -u +%Y%m%d_%H%M%S)
OUTDIR=gs://${BUCKET}/babyweight/tuned_$TIMESTAMP
JOB_NAME=babyweight_tuned_$TIMESTAMP

PYTHON_PACKAGE_URI=gs://${BUCKET}/babyweight/babyweight_trainer-0.1.tar.gz
PYTHON_PACKAGE_EXECUTOR_IMAGE_URI="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-8:latest"
PYTHON_MODULE=trainer.task

echo > ./tuned_config.yaml "workerPoolSpecs:
  machineSpec:
    machineType: n1-standard-8
  replicaCount: 1
  pythonPackageSpec:
    executorImageUri: $PYTHON_PACKAGE_EXECUTOR_IMAGE_URI
    packageUris: $PYTHON_PACKAGE_URI
    pythonModule: $PYTHON_MODULE
    args:
    - --train_data_path=gs://${BUCKET}/babyweight/data/train*.csv
    - --eval_data_path=gs://${BUCKET}/babyweight/data/eval*.csv
    - --output_dir=$OUTDIR
    - --num_epochs=10
    - --train_examples=20000
    - --eval_steps=100
    - --batch_size=32
    - --nembeds=8"
    
gcloud ai custom-jobs create \
  --region=${REGION} \
  --display-name=$JOB_NAME \
  --config=tuned_config.yaml

## ラボのまとめ: 
このラボでは、環境のセットアップ、ハイパーパラメータの引数解析コードを保持するトレーナー モジュールの task.py の作成、Keras モデルコードを保持するトレーナー モジュールの model.py の作成、トレーナー モジュール パッケージのローカルでの実行、および Vertex AI へのトレーニング ジョブの送信を行いました。

Copyright 2021 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.